In [41]:
import json
import pandas as pd
import os

In [42]:
def extract_lines_from_segement(feature):
    temp_lines = pd.DataFrame({
        'line': [line['name'] for line in feature['properties']['lines']]
    }) 
    temp_lines['segmentId'] = feature['properties']['id']
    return temp_lines

In [43]:
def extract_segment_geometry(segment):
    for feature in raw_lines['features']:
        if feature['properties']['id'] == segment:
            return feature['geometry']['coordinates']

In [44]:
def concat_segments(segments):
    geom = []
    for segment in segments:
        geom.extend(extract_segment_geometry(segment))
    return geom

In [45]:
def reverse_coords(line_string):
    return [coords[::-1] for coords in line_string]

In [46]:
with open('../data/raw_lines.json', 'r') as in_file:
    raw_lines = json.load(in_file)

In [47]:
lines = pd.concat([extract_lines_from_segement(feature) for feature in raw_lines['features']])
lines['line'] = lines['line'].apply(lambda x: x.replace('line', '').strip())

In [48]:
lines = lines.groupby('line').agg({'segmentId': list}).reset_index()
lines['geometry'] = lines['segmentId'].apply(concat_segments)

In [49]:
features = [{
    'type': 'Feature',
    'properties': {
        'line': lines['line'][i]
    },
    'geometry': {
        'type': 'LineString',
        'coordinates': reverse_coords(lines['geometry'][i])
    }
} for i in range(0, len(lines))]

In [50]:
out_data = {
    "type": "FeatureCollection",
    "name": "tube-lines-simplified",
    "features": features
}

In [51]:
with open('../data/lines.json', 'w') as out_file:
    json.dump(out_data, out_file)